# Assay curation audit with row-loss tracking

Reproduces the curation in [`dataset_curation.ipynb`](dataset_curation.ipynb) and the
`kinodata` loader in [`dti/data.py`](../dti/data.py), but records **filter name /
dataset size after the filter / dropped rows** for every single step, for all three
benchmark datasets: `landrum`, `kinodata` and `omnivore`.

## How it works

1. Export one **broad** candidate table from SQL, filtered only on `pchembl_value IS NOT NULL`
   (the label has to exist). Everything else that `gather_data` does in SQL is re-expressed as
   an auditable Polars filter, so no curation decision is hidden inside the base query.
2. Apply the filters as a chain of *cumulative boolean masks* and count rows and assays
   surviving each prefix. This needs a **single streaming pass** over the base table instead of
   one collect + one `sink_parquet` per step.
3. `kinodata` is not produced by `gather_data` — it is an external ChEMBL 33 export
   (`activities-chembl33_v0.5.csv`, kinodata-3D). Its audit chain therefore mirrors
   `dti.data.load_kinodata` instead.
4. Validate the final row counts against the curated files actually used for training
   (`landrum.csv`, `omnivore.csv`, `activities-chembl33_v0.5.csv`).

## Faithfulness notes

Points where the naive translation of `gather_data` into Polars silently disagrees with the SQL:

- **Assay size is computed *before* the doc/mutant/confidence pruning.** In `gather_data`, `cnt`
  is `COUNT(DISTINCT molregno)` inside the `temp_assays` `GROUP BY`; the later `DELETE`s remove
  assays but never recompute `cnt`. Recomputing it after those steps gives a different assay set.
- **Assay size counts distinct compounds, not rows.** Assays with repeated measurements of the
  same molecule can end up with more rows than `max_assay_size`.
- **`LOWER(description) LIKE '%mutant%'` keeps rows with a `NULL` description** (`NULL LIKE` is
  `NULL`, so the `DELETE` does not fire). Polars' `~str.contains(...)` propagates null and would
  *drop* them, so the mask needs `.fill_null(False)`.
- **`JOIN docs` is an inner join**, so assays with no `docs` row are dropped for every dataset,
  independently of `onlyDocs`. It is tracked as its own step.
- **The final `goldilocks` query drops rows twice more**: once by joining `compound_structures`
  (no SMILES) and once by requiring the target's component sequence to be globally unique
  (`HAVING COUNT(*) = 1`). Both are real, sizeable losses and are audited here.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
import polars as pl

pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(60)

DB_PATH = Path("../data/chembl_32/chembl_32_sqlite/chembl_32.db")
DATA_DIR = Path("../data")
BASE_DIR = DATA_DIR / "curation_audit"
BASE_DIR.mkdir(parents=True, exist_ok=True)

BASE_PATH = BASE_DIR / "00_base_activities.parquet"
SEQ_PATH = BASE_DIR / "00_unique_sequence_targets.parquet"

# curated files produced by the original pipeline, used to validate the audit
CURATED_FILES = {
    "landrum": DATA_DIR / "landrum.csv",
    "omnivore": DATA_DIR / "omnivore.csv",
}
KINODATA_PATH = DATA_DIR / "activities-chembl33_v0.5.csv"

## Step 1 — broad base export

Only `pchembl_value IS NOT NULL` is applied in SQL. All joins are `LEFT` joins so that the
inner-join semantics of the original query (`docs`, `target_dictionary`, `compound_structures`)
become explicit, auditable filter steps rather than invisible row loss.

The unique-component-sequence target map is exported separately: it reproduces the
`sequence IN (... HAVING COUNT(*) = 1)` predicate of the final `goldilocks` query as a `tid`
allow-list.

In [2]:
BASE_QUERY = """
SELECT
    a.activity_id,
    a.assay_id,
    a.molregno,
    ass.tid,
    ass.doc_id,
    docs.doc_id      AS doc_row_id,
    docs.year        AS doc_year,
    a.pchembl_value,
    a.standard_type,
    a.standard_units,
    a.standard_relation,
    a.data_validity_comment,
    ass.description,
    ass.variant_id,
    ass.confidence_score,
    td.target_type,
    cs.canonical_smiles
FROM activities a
JOIN assays ass USING (assay_id)
LEFT JOIN docs ON ass.doc_id = docs.doc_id
LEFT JOIN target_dictionary td USING (tid)
LEFT JOIN compound_structures cs USING (molregno)
WHERE a.pchembl_value IS NOT NULL;
"""

# targets whose component sequence occurs exactly once in component_sequences
UNIQUE_SEQ_QUERY = """
SELECT tc.tid, cs.component_id, cs.sequence AS component_sequence
FROM target_components tc
JOIN component_sequences cs ON tc.component_id = cs.component_id
WHERE cs.sequence IN (
    SELECT sequence FROM component_sequences GROUP BY sequence HAVING COUNT(*) = 1
);
"""


def export_base(force: bool = False) -> None:
    """Export the broad candidate table once; re-use the parquet afterwards."""
    if BASE_PATH.exists() and SEQ_PATH.exists() and not force:
        print(f"re-using cached {BASE_PATH.name} and {SEQ_PATH.name}")
        return

    import sqlite3

    con = sqlite3.connect(DB_PATH)
    try:
        base = pd.read_sql(BASE_QUERY, con)
        seqs = pd.read_sql(UNIQUE_SEQ_QUERY, con)
    finally:
        con.close()

    pl.from_pandas(base).write_parquet(BASE_PATH)
    pl.from_pandas(seqs).write_parquet(SEQ_PATH)
    print(f"base {base.shape}, unique-sequence targets {seqs.shape}")


export_base()

base_lf = pl.scan_parquet(BASE_PATH)
seq_map = pl.read_parquet(SEQ_PATH)
unique_seq_tids = seq_map["tid"].unique().to_list()

print(f"{base_lf.select(pl.len()).collect().item():,} base activities")
print(f"{len(unique_seq_tids):,} targets with a globally unique component sequence")

base (3950349, 17), unique-sequence targets (13492, 3)
3,950,349 base activities
10,256 targets with a globally unique component sequence


## Step 2 — the audit engine

`audit_chain` takes `(filter_name, predicate)` pairs, builds the cumulative masks `m1`,
`m1 & m2`, `m1 & m2 & m3`, ... and evaluates all of their row and assay counts in one `select`.
Polars therefore reads the base table once regardless of how many steps there are.

In [3]:
def audit_chain(
    lf: pl.LazyFrame,
    steps: list[tuple[str, pl.Expr]],
    *,
    assay_col: str = "assay_id",
    base_label: str = "base",
) -> pl.DataFrame:
    """
    Apply filters cumulatively and report, for every step:

    - ``filter_name``
    - ``dataset_size_after_filter``
    - ``dropped_rows``

    plus the surviving assay count and the fraction of the base that is left.
    """
    exprs = [
        pl.len().alias("n_00"),
        pl.col(assay_col).n_unique().alias("a_00"),
    ]
    cum: pl.Expr | None = None
    for i, (_, predicate) in enumerate(steps, start=1):
        cum = predicate if cum is None else (cum & predicate)
        exprs.append(cum.sum().alias(f"n_{i:02d}"))
        exprs.append(pl.col(assay_col).filter(cum).n_unique().alias(f"a_{i:02d}"))

    counts = lf.select(exprs).collect().row(0, named=True)

    names = [base_label] + [name for name, _ in steps]
    base_n = counts["n_00"]
    rows, previous = [], None
    for i, name in enumerate(names):
        n = counts[f"n_{i:02d}"]
        rows.append(
            {
                "step": i,
                "filter_name": name,
                "dataset_size_after_filter": n,
                "dropped_rows": 0 if previous is None else previous - n,
                "n_assays_after_filter": counts[f"a_{i:02d}"],
                "frac_of_base": n / base_n if base_n else float("nan"),
            }
        )
        previous = n
    return pl.DataFrame(rows)


AUDIT_COLUMNS = ["filter_name", "dataset_size_after_filter", "dropped_rows"]

## Step 3 — the ChEMBL 32 chain (`landrum`, `omnivore`)

The chain is split at `N_POOL_STEPS`: the first steps define the **activity pool** over which
`gather_data` computes `cnt = COUNT(DISTINCT molregno)` per assay. The assay-size filter uses
that pool count, *not* a count recomputed after the doc/mutant/confidence pruning.

In [4]:
MUTANT_PATTERN = "mutant|mutation|variant"

# NULL description must survive, matching `LOWER(description) LIKE '%mutant%'` in SQL
mutant_assay = (
    pl.col("description")
    .str.to_lowercase()
    .str.contains(MUTANT_PATTERN)
    .fill_null(False)
)


def pool_steps(standard_type: str) -> list[tuple[str, pl.Expr]]:
    """Filters from the `temp_assays` WHERE clause, i.e. before `cnt` is computed."""
    return [
        (
            f"standard_type == '{standard_type}'",
            pl.col("standard_type") == standard_type,
        ),
        ("standard_units == 'nM'", pl.col("standard_units") == "nM"),
        ("data_validity_comment is null", pl.col("data_validity_comment").is_null()),
        ("standard_relation == '='", pl.col("standard_relation") == "="),
        ("target_type == 'SINGLE PROTEIN'", pl.col("target_type") == "SINGLE PROTEIN"),
        # `JOIN docs` is an inner join, so this applies even when only_docs=False
        ("assay has a docs record", pl.col("doc_row_id").is_not_null()),
    ]


N_POOL_STEPS = len(pool_steps("IC50"))


def chembl_steps(
    standard_type: str,
    *,
    min_assay_size: int,
    max_assay_size: float,
    only_docs: bool,
    remove_mutants: bool,
    only_high_confidence: bool,
    assay_ids_in_size_range: list[int],
) -> list[tuple[str, pl.Expr]]:
    steps = pool_steps(standard_type)

    # the DELETE / confidence steps on temp_assays
    if only_docs:
        steps.append(("doc_year is not null", pl.col("doc_year").is_not_null()))
    if remove_mutants:
        steps.append(
            (
                "no mutant / variant assay",
                pl.col("variant_id").is_null() & ~mutant_assay,
            )
        )
    if only_high_confidence:
        steps.append(("confidence_score == 9", pl.col("confidence_score") == 9))

    # the goldilocks table
    steps.append(
        (
            f"assay size in [{min_assay_size:g}, {max_assay_size:g}] compounds",
            pl.col("assay_id").is_in(assay_ids_in_size_range),
        )
    )
    steps.append(("has canonical_smiles", pl.col("canonical_smiles").is_not_null()))
    steps.append(
        ("target has unique component sequence", pl.col("tid").is_in(unique_seq_tids))
    )
    return steps


def assay_sizes(standard_type: str) -> pl.DataFrame:
    """`cnt` per assay: distinct compounds in the activity pool."""
    lf = base_lf
    for _, predicate in pool_steps(standard_type):
        lf = lf.filter(predicate)
    return (
        lf.group_by("assay_id")
        .agg(pl.col("molregno").n_unique().alias("assay_size"))
        .collect()
    )


def audit_chembl_dataset(
    name: str, **config
) -> tuple[pl.DataFrame, list[tuple[str, pl.Expr]]]:
    sizes = assay_sizes(config["standard_type"])
    in_range = sizes.filter(
        (pl.col("assay_size") >= config["min_assay_size"])
        & (pl.col("assay_size") <= config["max_assay_size"])
    )["assay_id"].to_list()

    steps = chembl_steps(assay_ids_in_size_range=in_range, **config)
    audit = audit_chain(
        base_lf, steps, base_label="base: pchembl_value is not null"
    ).with_columns(pl.lit(name).alias("dataset"))
    return audit, steps

## Step 4 — the `kinodata` chain

`kinodata` never touches the ChEMBL 32 SQLite file. It is a pre-built ChEMBL 33 export whose
curation happens in `dti.data.load_kinodata` + `dti.data._process`, so the audit follows those
four filters.

In [5]:
KINODATA_COLUMNS = {
    "type": "activities.standard_type",
    "activity": "activities.standard_value",
    "smiles": "compound_structures.canonical_smiles",
    "sequence": "component_sequences.sequence",
    "assay": "assays.chembl_id",
}


def audit_kinodata(
    name: str = "kinodata", activity_types: list[str] = ["pIC50"]
) -> tuple[pl.DataFrame, list[tuple[str, pl.Expr]]]:
    """Mirrors dti.data.load_kinodata (which then calls _process)."""
    c = KINODATA_COLUMNS
    steps = [
        (f"standard_type in {activity_types}", pl.col(c["type"]).is_in(activity_types)),
        ("has canonical_smiles", pl.col(c["smiles"]).is_not_null()),
        ("activity value is not null", pl.col(c["activity"]).is_not_null()),
        ("component sequence is not null", pl.col(c["sequence"]).is_not_null()),
    ]
    audit = audit_chain(
        pl.scan_csv(KINODATA_PATH),
        steps,
        assay_col=c["assay"],
        base_label=f"base: {KINODATA_PATH.name}",
    ).with_columns(pl.lit(name).alias("dataset"))
    return audit, steps

## Step 5 — run the audit for all three datasets

The `landrum` and `omnivore` configurations are the argument sets that
[`dataset_curation.ipynb`](dataset_curation.ipynb) passed to `gather_data` (`landrum` used the
function defaults).

In [6]:
CHEMBL_DATASETS = {
    "landrum": dict(
        standard_type="IC50",
        min_assay_size=20,
        max_assay_size=100,
        only_docs=True,
        remove_mutants=True,
        only_high_confidence=True,
    ),
    "omnivore": dict(
        standard_type="IC50",
        min_assay_size=2,
        max_assay_size=5e6,
        only_docs=False,
        remove_mutants=True,
        only_high_confidence=False,
    ),
}

audits: dict[str, pl.DataFrame] = {}
step_defs: dict[str, list[tuple[str, pl.Expr]]] = {}

for dataset_name, dataset_config in CHEMBL_DATASETS.items():
    audits[dataset_name], step_defs[dataset_name] = audit_chembl_dataset(
        dataset_name, **dataset_config
    )

audits["kinodata"], step_defs["kinodata"] = audit_kinodata()

filter_audit = pl.concat(
    [audits[d] for d in ("landrum", "kinodata", "omnivore")], how="vertical"
).select(["dataset", "step", *AUDIT_COLUMNS, "n_assays_after_filter", "frac_of_base"])

filter_audit.write_csv(BASE_DIR / "filter_audit.csv")
filter_audit

dataset,step,filter_name,dataset_size_after_filter,dropped_rows,n_assays_after_filter,frac_of_base
str,i64,str,i64,i64,i64,f64
"""landrum""",0,"""base: pchembl_value is not null""",3950349,0,262637,1.0
"""landrum""",1,"""standard_type == 'IC50'""",1724810,2225539,176048,0.436622
"""landrum""",2,"""standard_units == 'nM'""",1724767,43,176044,0.436611
"""landrum""",3,"""data_validity_comment is null""",1723575,1192,175932,0.43631
"""landrum""",4,"""standard_relation == '='""",1723413,162,175931,0.436269
"""landrum""",5,"""target_type == 'SINGLE PROTEIN'""",903061,820352,85396,0.228603
"""landrum""",6,"""assay has a docs record""",903061,0,85396,0.228603
"""landrum""",7,"""doc_year is not null""",843055,60006,84279,0.213413
"""landrum""",8,"""no mutant / variant assay""",817441,25614,81581,0.206929


### The requested table, one block per dataset

`filter_name` / `dataset_size_after_filter` / `dropped_rows`, in application order.

In [7]:
for dataset_name in ("landrum", "kinodata", "omnivore"):
    audit = audits[dataset_name]
    final = audit["dataset_size_after_filter"][-1]
    base = audit["dataset_size_after_filter"][0]
    print(
        f"=== {dataset_name}: {base:,} -> {final:,} rows ({final / base:.1%} retained) ==="
    )
    print(audit.select(AUDIT_COLUMNS))
    print()

=== landrum: 3,950,349 -> 259,823 rows (6.6% retained) ===
shape: (13, 3)
┌──────────────────────────────────────┬───────────────────────────┬──────────────┐
│ filter_name                          ┆ dataset_size_after_filter ┆ dropped_rows │
│ ---                                  ┆ ---                       ┆ ---          │
│ str                                  ┆ i64                       ┆ i64          │
╞══════════════════════════════════════╪═══════════════════════════╪══════════════╡
│ base: pchembl_value is not null      ┆ 3950349                   ┆ 0            │
│ standard_type == 'IC50'              ┆ 1724810                   ┆ 2225539      │
│ standard_units == 'nM'               ┆ 1724767                   ┆ 43           │
│ data_validity_comment is null        ┆ 1723575                   ┆ 1192         │
│ standard_relation == '='             ┆ 1723413                   ┆ 162          │
│ target_type == 'SINGLE PROTEIN'      ┆ 903061                    ┆ 820352       │
│ 

### Where each dataset loses the most rows

In [8]:
biggest_losses = (
    filter_audit.filter(pl.col("step") > 0)
    .sort(["dataset", "dropped_rows"], descending=[False, True])
    .group_by("dataset", maintain_order=True)
    .head(3)
    .select(["dataset", "filter_name", "dropped_rows", "dataset_size_after_filter"])
)
biggest_losses

dataset,filter_name,dropped_rows,dataset_size_after_filter
str,str,i64,i64
"""kinodata""","""standard_type in ['pIC50']""",31472,180135
"""kinodata""","""has canonical_smiles""",153,179982
"""kinodata""","""component sequence is not null""",0,179982
"""landrum""","""standard_type == 'IC50'""",2225539,1724810
"""landrum""","""target_type == 'SINGLE PROTEIN'""",820352,903061
"""landrum""","""assay size in [20, 100] compounds""",302361,263555
"""omnivore""","""standard_type == 'IC50'""",2225539,1724810
"""omnivore""","""target_type == 'SINGLE PROTEIN'""",820352,903061
"""omnivore""","""assay size in [2, 5e+06] compounds""",27494,848950


## Step 6 — validate the audit against the curated files

The last row of each chain has to reproduce the file the training runs actually consume. If these
do not match, a filter in the chain is not equivalent to the original SQL.

In [9]:
checks = []
for dataset_name, path in CURATED_FILES.items():
    if not path.exists():
        print(f"skipping {dataset_name}: {path} not found")
        continue
    actual = pl.scan_csv(path).select(pl.len()).collect().item()
    checks.append(
        {
            "dataset": dataset_name,
            "audited_rows": audits[dataset_name]["dataset_size_after_filter"][-1],
            "rows_in_curated_file": actual,
        }
    )

if KINODATA_PATH.exists():
    c = KINODATA_COLUMNS
    actual = (
        pl.scan_csv(KINODATA_PATH)
        .filter(
            pl.col(c["type"]).is_in(["pIC50"])
            & pl.col(c["smiles"]).is_not_null()
            & pl.col(c["activity"]).is_not_null()
            & pl.col(c["sequence"]).is_not_null()
        )
        .select(pl.len())
        .collect()
        .item()
    )
    checks.append(
        {
            "dataset": "kinodata",
            "audited_rows": audits["kinodata"]["dataset_size_after_filter"][-1],
            "rows_in_curated_file": actual,
        }
    )

validation = pl.DataFrame(checks).with_columns(
    (pl.col("audited_rows") == pl.col("rows_in_curated_file")).alias("matches")
)
assert validation["matches"].all(), validation
validation

dataset,audited_rows,rows_in_curated_file,matches
str,i64,i64,bool
"""landrum""",259823,259823,true
"""omnivore""",838155,838155,true
"""kinodata""",179982,179982,true


## Step 7 — materialise a curated dataset (optional)

The audit itself never writes intermediate parquet files, since the cumulative-mask pass makes
them unnecessary. This helper re-applies a chain and writes the final frame when the curated
table itself is wanted.

In [10]:
def materialise(dataset_name: str, out_path: Path | None = None) -> Path:
    """Re-apply a dataset's chain and write the final curated table to parquet."""
    if dataset_name == "kinodata":
        lf = pl.scan_csv(KINODATA_PATH)
    else:
        lf = base_lf
    for _, predicate in step_defs[dataset_name]:
        lf = lf.filter(predicate)

    out_path = out_path or BASE_DIR / f"curated_{dataset_name}.parquet"
    lf.sink_parquet(out_path)
    print(
        f"wrote {out_path} ({pl.scan_parquet(out_path).select(pl.len()).collect().item():,} rows)"
    )
    return out_path


# materialise("landrum")

## Notes

- `data/curation_audit/` may still contain `0*_keep_*.parquet` / `final_curated_dataset.csv` from
  the earlier version of this notebook. Those were produced by a chain that computed assay sizes
  *after* the doc/mutant/confidence filters and skipped the SMILES and unique-sequence joins, so
  their numbers do not reproduce `landrum.csv` / `omnivore.csv` and they can be deleted.
- `landrum_large` (`min_assay_size=5`, `max_assay_size=1e6`, all curation flags on) is also
  produced by `dataset_curation.ipynb`. It is not one of the three benchmark datasets, but adding
  it is a one-line addition to `CHEMBL_DATASETS`.